# L24 · 微调入门：教 AI 说你的语言

**学习目标**
- 理解「微调（Fine-tuning）」：在通用模型上，用你的数据再训练
- 区分「预训练（学通用）」与「微调（学专属）」
- 亲手用你的数据训练一个「文本分类器」（离线，无需 GPU）

**前置依赖**：L12（ML 流程）、L22（向量化）、L19（大模型基础）  
**预计时长**：50 分钟  
**技术栈**：`scikit-learn`、`numpy`（TF-IDF + 逻辑回归，离线可运行）

---

## 概念讲解：微调 = 给通才做「岗前培训」

大模型先在海量文本上「预训练」，成了什么都懂点的通才。
但你的公司要它**只精准做一件事**（如：判断工单紧急程度），就得用你的标注数据「再练一遍」——这就是**微调**。

本课我们用一个小模型演示「微调」的本质：**用专属数据调整参数，让它在你的任务上变准**。
（真实 LLM 微调用 GPU + 亿级参数，原理一致，本课让你先吃透思想。）

## 第一步：准备「你的专属数据」

In [ ]:
# 你标注的客服工单：0=普通，1=紧急
texts = [
    "我想问问发货时间", "怎么退货", "发票怎么开",
    "系统崩了全公司无法登录", "数据全丢了快来人", "服务器着火了",
    "咨询会员价格", "修改收货地址", "账户被锁进不去",
]
labels = [0, 0, 0, 1, 1, 1, 0, 0, 1]
print("数据集：", len(texts), "条，紧急", sum(labels), "条")

## 第二步：用你的数据「微调」一个分类器（TF-IDF + 逻辑回归）

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

# 这就是「微调」：在数据上拟合，调整内部参数
model = make_pipeline(TfidfVectorizer(), LogisticRegression())
model.fit(texts, labels)
print("✅ 微调完成（模型参数已根据你的数据调整）")

## 第三步：看它学会了你的标准

In [ ]:
for t in ["我的订单到哪了", "机房断电整个网站挂了"]:
    pred = model.predict([t])[0]
    print(f"  「{t}」 → 判为 {'紧急' if pred==1 else '普通'}")

# 🎯 AHA 顿悟单元格：你的「工单紧急度分类器」上线

运行下面代码。你会看到：模型在你**没标过**的新句子上，也能按你教的标准判断紧急程度。
改 `new_tickets` 加你自己的句子，看它如何「举一反三」。

> 你刚刚完成的，就是真实「微调」的迷你版：用你的数据，让模型学会你的规则。
> 大厂用同样思路，把通用 GPT 微调成「医疗顾问」「法律助手」「客服专家」。

In [ ]:
# ===== 运行我！看微调后的模型判断新工单 =====
new_tickets = [
    "请问怎么开发票",
    "数据库被勒索病毒加密了",
    "想升级套餐",
    "所有用户都无法支付紧急",
]
print("  🎫 你的『工单紧急度分类器』（已用你的数据微调）开始工作：\n")
for t in new_tickets:
    pred = model.predict([t])[0]
    proba = model.predict_proba([t])[0][pred]
    tag = "🚨 紧急" if pred == 1 else "🟢 普通"
    print(f"  {tag}  (置信度 {proba:.0%})  {t}")
print("\n  ✨ 它从你的 9 条标注里学会了标准，还能判断没见过的新工单！")

# 📝 讲师备课笔记（接手 Agent 专用）

**本课难点**：须澄清「本课是微调思想的轻量投影」，真 LLM 微调改的是 Transformer 权重（SFT/PEFT），非 sklearn；但「用专属数据调整参数→任务变准」的本质一致，零依赖保证可运行。  
**易错点**：样本极少（9 条）过拟合，但本课重在演示「思想」而非精度；`predict_proba` 需 pipeline 支持。  
**AHA 机制**：新句子正确分类+置信度，强「模型学会我的标准」实感。  
**衔接**：L31 SFT（真后训练）；L33 DPO；L39 后训练管线。  
**真 LLM 路径说明**：在备课笔记给方向——用 `transformers`+`Trainer` 或 `peft` LoRA 微调小模型（如 TinyLlama），需 GPU/Colab，列为进阶实践。  
**依赖**：`pip install scikit-learn numpy`。

# 📚 作业 / 下一步

1. 在 `texts/labels` 里加几条你自己的工单，重新微调看变化。
2. 搜索「LoRA 微调」「SFT」了解真大模型微调。
3. 进入 **阶段五 · 工程化约束 AI**：L25 AI 评测体系 —— 怎么知道你调出来的 AI 到底好不好？